In [1]:
import pandas as pd
df=pd.read_csv("../data/processed/us_accidents_cleaned.csv")
df.shape

(500000, 38)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 38 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Source                 500000 non-null  object 
 1   Severity               500000 non-null  int64  
 2   Start_Time             500000 non-null  object 
 3   End_Time               500000 non-null  object 
 4   Start_Lat              500000 non-null  float64
 5   Start_Lng              500000 non-null  float64
 6   Distance(mi)           500000 non-null  float64
 7   City                   500000 non-null  object 
 8   County                 500000 non-null  object 
 9   State                  500000 non-null  object 
 10  Timezone               500000 non-null  object 
 11  Airport_Code           500000 non-null  object 
 12  Weather_Timestamp      500000 non-null  object 
 13  Temperature(F)         500000 non-null  float64
 14  Wind_Chill(F)          500000 non-nu

In [4]:
df.select_dtypes(include="object").columns.tolist()

['Source',
 'Start_Time',
 'End_Time',
 'City',
 'County',
 'State',
 'Timezone',
 'Airport_Code',
 'Weather_Timestamp',
 'Wind_Direction',
 'Weather_Condition',
 'Sunrise_Sunset',
 'Civil_Twilight',
 'Nautical_Twilight',
 'Astronomical_Twilight']

In [5]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"], format="mixed")
df["End_Time"] = pd.to_datetime(df["End_Time"], format="mixed")
df["Weather_Timestamp"] = pd.to_datetime(df["Weather_Timestamp"], format="mixed")

In [6]:
df[
    ["Start_Time",
     "End_Time",
     "Weather_Timestamp"]
].dtypes

Start_Time           datetime64[ns]
End_Time             datetime64[ns]
Weather_Timestamp    datetime64[ns]
dtype: object

In [7]:
df["Hour"] = df["Start_Time"].dt.hour

df["Month"] = df["Start_Time"].dt.month

df["DayOfWeek"] = df["Start_Time"].dt.dayofweek

df["Year"] = df["Start_Time"].dt.year

df["Is_Weekend"] = (
    df["DayOfWeek"] >= 5
).astype(int)

df["Accident_Duration_Minutes"] = (
    df["End_Time"] -
    df["Start_Time"]
).dt.total_seconds() / 60

In [8]:
df["Is_Daytime"] = (
    df["Sunrise_Sunset"] == "Day"
).astype(int)

In [9]:
df["Rush_Hour"] = (
    (
        (df["Hour"] >= 7) &
        (df["Hour"] <= 9)
    )
    |
    (
        (df["Hour"] >= 16) &
        (df["Hour"] <= 18)
    )
).astype(int)

In [10]:
df.shape

(500000, 46)

In [11]:
df[
    [
        "Hour",
        "Month",
        "DayOfWeek",
        "Year",
        "Is_Weekend",
        "Is_Daytime",
        "Rush_Hour"
    ]
].head()

,Hour,Month,DayOfWeek,Year,Is_Weekend,Is_Daytime,Rush_Hour
0,9,4,4,2020,0,1,1
1,10,4,3,2022,0,1,0
2,16,8,4,2016,0,1,1
3,15,9,4,2019,0,1,0
4,16,6,0,2019,0,1,1


In [12]:
columns_to_drop = [
    "City",
    "County",
    "Airport_Code"
]

df = df.drop(columns=columns_to_drop)

In [13]:
datetime_columns = [
    "Start_Time",
    "End_Time",
    "Weather_Timestamp"
]

df = df.drop(columns=datetime_columns)

In [14]:
df.shape

(500000, 40)

In [15]:
df.select_dtypes(include="object").columns.tolist()

['Source',
 'State',
 'Timezone',
 'Wind_Direction',
 'Weather_Condition',
 'Sunrise_Sunset',
 'Civil_Twilight',
 'Nautical_Twilight',
 'Astronomical_Twilight']

In [16]:
for col in df.select_dtypes(include="object").columns:
    print(col, ":", df[col].nunique())

Source : 3
State : 49
Timezone : 4
Wind_Direction : 24
Weather_Condition : 105
Sunrise_Sunset : 2
Civil_Twilight : 2
Nautical_Twilight : 2
Astronomical_Twilight : 2


In [18]:
from sklearn.preprocessing import LabelEncoder

In [20]:
le=LabelEncoder()

In [21]:
binary_cols = [
    "Sunrise_Sunset",
    "Civil_Twilight",
    "Nautical_Twilight",
    "Astronomical_Twilight"
]

for col in binary_cols:
    df[col] = le.fit_transform(df[col])

In [22]:
categorical_cols = [
    "Source",
    "State",
    "Timezone",
    "Wind_Direction",
    "Weather_Condition"
]

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 40 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Source                     500000 non-null  int64  
 1   Severity                   500000 non-null  int64  
 2   Start_Lat                  500000 non-null  float64
 3   Start_Lng                  500000 non-null  float64
 4   Distance(mi)               500000 non-null  float64
 5   State                      500000 non-null  int64  
 6   Timezone                   500000 non-null  int64  
 7   Temperature(F)             500000 non-null  float64
 8   Wind_Chill(F)              500000 non-null  float64
 9   Humidity(%)                500000 non-null  float64
 10  Pressure(in)               500000 non-null  float64
 11  Visibility(mi)             500000 non-null  float64
 12  Wind_Direction             500000 non-null  int64  
 13  Wind_Speed(mph)            50

In [24]:
df.select_dtypes(include="object").columns

Index([], dtype='object')

In [25]:
df["Severity"].value_counts()

Severity
2    397765
3     84543
4     13309
1      4383
Name: count, dtype: int64

In [26]:
df["Severity"].value_counts(normalize=True)*100

Severity
2    79.5530
3    16.9086
4     2.6618
1     0.8766
Name: proportion, dtype: float64

In [27]:
df.to_csv(
    "../data/processed/us_accidents_feature_engineered.csv",
    index=False
)

In [29]:
df["Severity"].value_counts()

Severity
2    397765
3     84543
4     13309
1      4383
Name: count, dtype: int64

In [30]:
df["Severity"].value_counts(normalize=True) * 100

Severity
2    79.5530
3    16.9086
4     2.6618
1     0.8766
Name: proportion, dtype: float64

In [31]:
df.to_csv(
    "../data/processed/us_accidents_feature_engineered.csv",
    index=False
)

In [32]:
df.shape

(500000, 40)

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/us_accidents_cleaned.csv"
)

In [2]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"], format="mixed")
df["End_Time"] = pd.to_datetime(df["End_Time"], format="mixed")
df["Weather_Timestamp"] = pd.to_datetime(df["Weather_Timestamp"], format="mixed")

df["Hour"] = df["Start_Time"].dt.hour
df["Month"] = df["Start_Time"].dt.month
df["DayOfWeek"] = df["Start_Time"].dt.dayofweek
df["Year"] = df["Start_Time"].dt.year

df["Is_Weekend"] = (df["DayOfWeek"] >= 5).astype(int)

df["Accident_Duration_Minutes"] = (
    df["End_Time"] - df["Start_Time"]
).dt.total_seconds() / 60

df["Is_Daytime"] = (
    df["Sunrise_Sunset"] == "Day"
).astype(int)

df["Rush_Hour"] = (
    ((df["Hour"] >= 7) & (df["Hour"] <= 9))
    |
    ((df["Hour"] >= 16) & (df["Hour"] <= 18))
).astype(int)

In [3]:
df = df.drop(columns=[
    "City",
    "County",
    "Airport_Code",
    "Start_Time",
    "End_Time",
    "Weather_Timestamp"
])

In [4]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

categorical_cols = [
    "Source",
    "State",
    "Timezone",
    "Wind_Direction",
    "Weather_Condition"
]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

    encoders[col] = le

In [5]:
import joblib

joblib.dump(
    encoders,
    "../models/encoders.pkl"
)

['../models/encoders.pkl']

In [6]:
loaded_encoders = joblib.load(
    "../models/encoders.pkl"
)

print(loaded_encoders.keys())

dict_keys(['Source', 'State', 'Timezone', 'Wind_Direction', 'Weather_Condition'])


In [7]:
print(df["Weather_Condition"].unique()[:20])

print(df["Wind_Direction"].unique())

print(encoders["State"].classes_[:10])

[58 63  5 11 48 29  4 61  6 24 40 13 75 30 67 96 20 47 65 76]
[ 4 14 17 21 22 10  2  9 16  8  0 12  3 18 13 23  7  1 11 20  5  6 15 19]
['AL' 'AR' 'AZ' 'CA' 'CO' 'CT' 'DC' 'DE' 'FL' 'GA']


In [8]:
import joblib

encoders = joblib.load("../models/encoders.pkl")

In [9]:
print(df["Weather_Condition"].unique()[:20])

print(df["Wind_Direction"].unique())

print(encoders["State"].classes_[:10])

[58 63  5 11 48 29  4 61  6 24 40 13 75 30 67 96 20 47 65 76]
[ 4 14 17 21 22 10  2  9 16  8  0 12  3 18 13 23  7  1 11 20  5  6 15 19]
['AL' 'AR' 'AZ' 'CA' 'CO' 'CT' 'DC' 'DE' 'FL' 'GA']


In [10]:
import joblib

encoders = joblib.load("../models/encoders.pkl")

print(encoders["Weather_Condition"].classes_)

print(encoders["Wind_Direction"].classes_)

['Blowing Dust' 'Blowing Dust / Windy' 'Blowing Snow'
 'Blowing Snow / Windy' 'Clear' 'Cloudy' 'Cloudy / Windy'
 'Drifting Snow / Windy' 'Drizzle' 'Drizzle / Windy' 'Drizzle and Fog'
 'Fair' 'Fair / Windy' 'Fog' 'Fog / Windy' 'Freezing Drizzle'
 'Freezing Rain' 'Freezing Rain / Windy' 'Funnel Cloud' 'Hail' 'Haze'
 'Haze / Windy' 'Heavy Drizzle' 'Heavy Freezing Drizzle' 'Heavy Rain'
 'Heavy Rain / Windy' 'Heavy Sleet' 'Heavy Snow' 'Heavy Snow / Windy'
 'Heavy T-Storm' 'Heavy T-Storm / Windy' 'Heavy Thunderstorms and Rain'
 'Ice Pellets' 'Light Drizzle' 'Light Drizzle / Windy'
 'Light Freezing Drizzle' 'Light Freezing Fog' 'Light Freezing Rain'
 'Light Freezing Rain / Windy' 'Light Ice Pellets' 'Light Rain'
 'Light Rain / Windy' 'Light Rain Shower' 'Light Rain Showers'
 'Light Rain with Thunder' 'Light Sleet' 'Light Sleet / Windy'
 'Light Snow' 'Light Snow / Windy' 'Light Snow Grains' 'Light Snow Shower'
 'Light Snow Showers' 'Light Snow and Sleet'
 'Light Snow and Sleet / Windy' 'Light 

In [11]:
print(encoders["Source"].classes_)

print(encoders["Timezone"].classes_)

print(encoders["State"].classes_)

['Source1' 'Source2' 'Source3']
['US/Central' 'US/Eastern' 'US/Mountain' 'US/Pacific']
['AL' 'AR' 'AZ' 'CA' 'CO' 'CT' 'DC' 'DE' 'FL' 'GA' 'IA' 'ID' 'IL' 'IN'
 'KS' 'KY' 'LA' 'MA' 'MD' 'ME' 'MI' 'MN' 'MO' 'MS' 'MT' 'NC' 'ND' 'NE'
 'NH' 'NJ' 'NM' 'NV' 'NY' 'OH' 'OK' 'OR' 'PA' 'RI' 'SC' 'SD' 'TN' 'TX'
 'UT' 'VA' 'VT' 'WA' 'WI' 'WV' 'WY']
